In [1]:
import pandas as pd
import time # running the script again after a certain amount of time
import os # load env variables  
from datetime import datetime # datetime awareness
from opensky_api import OpenSkyApi # opensky api for data collection
from dotenv import load_dotenv # loading client_id and client_secret

In [2]:
def get_credentials() -> tuple[str, str]:
    """
    Returns credentials for OpenSky api.
    """
    load_dotenv() # load env variables
    username = os.getenv('clientId') # get opensky username
    password = os.getenv('clientSecret') # get opensky password
    return username, password

username, password = get_credentials()

In [3]:
api = OpenSkyApi(username=username, password=password)

In [4]:
save_directory = r"../data/"
os.makedirs(save_directory, exist_ok=True)
filename = os.path.join(save_directory, 'opensky_raw.csv')
filename

'../data/opensky_raw.csv'

In [5]:
target_rows = 1000000
total_rows_collected = 0
pulls = 0

while total_rows_collected < target_rows:
    try: 
        current_time = datetime.now().strftime('%H:%M:%S')
        print(f"[{current_time}] Pinging OpenSky api for flights...")
        
        states = api.get_states()
        
        if states is not None and states.states is not None:
            vector_data = []
            for s in states.states:
                vector_data.append([
                    s.icao24, s.callsign.strip() if s.callsign else None, s.origin_country,
                    s.time_position, s.longitude, s.latitude, s.baro_altitude,
                    s.on_ground, s.velocity, s.true_track, s.vertical_rate,
                    s.geo_altitude, s.category
                ])
                
            columns = ['icao24', 'callsign', 'origin_country', 
                        'timestamp', 'longitude', 'latitude', 'baro_altitude', 
                        'on_ground', 'velocity', 'true_track', 'vertical_rate', 
                        'geo_altitude', 's.category']
                
            df = pd.DataFrame(vector_data, columns=columns)
            
            rows_in_pull = len(df)
            total_rows_collected += rows_in_pull
            pulls += 1
            
            file_exists = os.path.isfile(filename)
            df.to_csv(filename, mode='a', header=not file_exists, index=False)
            
            percent_done = (total_rows_collected / target_rows) * 100
            print(f"Success! Added {rows_in_pull} rows.")
            print(f"Progress: {total_rows_collected:,} / {target_rows:,} ({percent_done:.2f}% complete)")
            
            if total_rows_collected >= target_rows:
                print(f"1,000,000 rows collected, shutting down.")
                break
                
        else:
            print("No aircrafts found.")
                
    except Exception as e:
        print(f"Network glitch: {e}")
        
    if total_rows_collected < target_rows:
        print("Sleeping 10 secs")
        time.sleep(10)

[22:00:04] Pinging OpenSky api for flights...
Success! Added 11169 rows.
Progress: 11,169 / 1,000,000 (1.12% complete)
Sleeping 10 secs
[22:00:17] Pinging OpenSky api for flights...
Success! Added 11171 rows.
Progress: 22,340 / 1,000,000 (2.23% complete)
Sleeping 10 secs
[22:00:30] Pinging OpenSky api for flights...
Success! Added 11182 rows.
Progress: 33,522 / 1,000,000 (3.35% complete)
Sleeping 10 secs
[22:00:44] Pinging OpenSky api for flights...
Success! Added 11181 rows.
Progress: 44,703 / 1,000,000 (4.47% complete)
Sleeping 10 secs
[22:00:56] Pinging OpenSky api for flights...
Success! Added 11184 rows.
Progress: 55,887 / 1,000,000 (5.59% complete)
Sleeping 10 secs
[22:01:15] Pinging OpenSky api for flights...
Success! Added 11171 rows.
Progress: 67,058 / 1,000,000 (6.71% complete)
Sleeping 10 secs
[22:01:29] Pinging OpenSky api for flights...
Success! Added 11178 rows.
Progress: 78,236 / 1,000,000 (7.82% complete)
Sleeping 10 secs
[22:01:43] Pinging OpenSky api for flights...
Su